## Setup

In [ ]:
import os
import sys

repo_path = '/Workspace/Users/dversch@emea.corpdir.net/.bundle/KG_Model/prod/files'
if repo_path not in sys.path:
    sys.path.append(repo_path)

import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import networkx as nx
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components, shortest_path
from rdflib import Graph as RDFGraph, RDF, RDFS, OWL, URIRef
from torch_geometric.data import Data

from training_data import Ontology
from node_prediction.gnn_tools import get_graph_mask

volume = "/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/"
out_dir = f"{volume}graph_geometry/"
os.makedirs(out_dir, exist_ok=True)

sulo_schema_path = f"{volume}sulo_kg.ttl"
sulo_data_path = f"{volume}subgraph_balanced_sulo_populated_graph.nt"

baseline_schema_path = f"{volume}base_kg.ttl"
baseline_data_path = f"{volume}subset_balanced_baseline_populated_graph.nt"

## Load Instance Graphs (.nt)

In [2]:
from types import SimpleNamespace

# Needs the TRUE (uncapped) edge_index/edge_type -- graph_sage.ipynb/RGCN.ipynb's
# {schema}_vars.pt is degree-capped for training, which would shift Table 1's
# structural metrics, so this keeps its own raw-edge cache. enum_nodes/
# enum_relations are still shared with {schema}_ontology_enums.pt.
def _load_or_build_ont(schema_label, nt_file_path, label_predicate, label_object_pattern):
    _raw_edges_path = f"{volume}{schema_label}_raw_edges.pt"
    _enum_cache_path = f"{volume}{schema_label}_ontology_enums.pt"
    if os.path.exists(_raw_edges_path) and os.path.exists(_enum_cache_path):
        _edges = torch.load(_raw_edges_path, map_location='cpu', weights_only=False)
        _enums = torch.load(_enum_cache_path, map_location='cpu', weights_only=False)
        ont = SimpleNamespace(
            edge_index=_edges['edge_index'], edge_type=_edges['edge_type'],
            num_nodes=_edges['num_nodes'],
            enum_nodes=_enums['enum_nodes'], enum_relations=_enums['enum_relations'],
        )
        print(f"[{schema_label}] Loaded cached raw edges + enums -- no .nt file access needed.")
        return ont

    ont = Ontology(
        nt_file_path=nt_file_path,
        label_predicate=label_predicate,
        label_object_pattern=label_object_pattern,
    )
    torch.save({
        'edge_index': ont.edge_index, 'edge_type': ont.edge_type, 'num_nodes': ont.num_nodes,
    }, _raw_edges_path)
    print(f"[{schema_label}] Saved raw edges: {_raw_edges_path}")
    if not os.path.exists(_enum_cache_path):
        torch.save({'enum_nodes': ont.enum_nodes, 'enum_relations': ont.enum_relations}, _enum_cache_path)
        print(f"[{schema_label}] Saved enums: {_enum_cache_path}")
    return ont

base_graph = _load_or_build_ont('base', baseline_data_path, "hasSurveyRating", r"SurveyRating/(\d+)")
sulo_graph = _load_or_build_ont('sulo', sulo_data_path, "hasFeature", r"SurveyRating/(\d+)")

# Lightweight PyG wrappers so gnn_tools.get_graph_mask can be reused here.
base_data = Data(edge_index=base_graph.edge_index, edge_type=base_graph.edge_type, num_nodes=base_graph.num_nodes)
sulo_data = Data(edge_index=sulo_graph.edge_index, edge_type=sulo_graph.edge_type, num_nodes=sulo_graph.num_nodes)

KGS = {
    'Ad-hoc (Base)': {'ont': base_graph, 'data': base_data},
    'SULO': {'ont': sulo_graph, 'data': sulo_data},
}

[base] Loaded cached raw edges + enums -- no .nt file access needed.
[sulo] Loaded cached raw edges + enums -- no .nt file access needed.

## Load Ontology Schemas (.ttl)

In [ ]:
def schema_metrics(ttl_path, upper_ontology_path=None):
    """
    Schema-level ontology-engineering metrics (RQ1a). Handles both files'
    typing conventions generically (sulo_kg.ttl: owl:Class/ObjectProperty/
    DatatypeProperty; base_kg.ttl: plain rdfs:Class/Property).

    Local-only counts always come strictly from `ttl_path` alone, for an
    apples-to-apples "how much did this project author itself" comparison.

    `upper_ontology_path`: sulo_kg.ttl's subClassOf edges point at
    w3id.org/sulo/ terms not typed locally -- without merging in the
    vendored copy (turtles/sulo.ttl) to resolve those parents, depth
    would wrongly read 0 (import-boundary artifact, not a flat taxonomy).
    Ad-hoc has no imports, so its true depth-0 needs no merge.
    """
    g = RDFGraph()
    g.parse(ttl_path, format="turtle")

    rdfs_ns = "http://www.w3.org/2000/01/rdf-schema#"
    class_types = [OWL.Class, URIRef(rdfs_ns + "Class")]
    generic_prop_types = [RDF.Property, URIRef(rdfs_ns + "Property")]

    classes = set()
    for t in class_types:
        classes |= set(g.subjects(RDF.type, t))

    object_props = set(g.subjects(RDF.type, OWL.ObjectProperty))
    datatype_props = set(g.subjects(RDF.type, OWL.DatatypeProperty))
    generic_props = set()
    for t in generic_prop_types:
        generic_props |= set(g.subjects(RDF.type, t))
    all_props = object_props | datatype_props | generic_props

    subclass_edges = [(c, p) for c, p in g.subject_objects(RDFS.subClassOf) if c in classes]

    # Depth traversal uses a separate, possibly-merged graph/class-set so the
    # local-only counts above are unaffected while depth_of() can still walk
    # parents that live in the (locally vendored) upper ontology.
    depth_triples = g
    known_classes = set(classes)
    if upper_ontology_path:
        upper_g = RDFGraph()
        upper_g.parse(upper_ontology_path, format="turtle")
        depth_triples = g + upper_g
        for t in class_types:
            known_classes |= set(upper_g.subjects(RDF.type, t))

    parent_of = {c: p for c, p in depth_triples.subject_objects(RDFS.subClassOf) if c in known_classes}

    def depth_of(cls, seen=None):
        seen = seen if seen is not None else set()
        if cls in seen:  # guard against accidental cycles
            return 0
        seen.add(cls)
        parent = parent_of.get(cls)
        # parent may also be a blank-node OWL restriction (e.g. SULO's
        # `rdfs:subClassOf [ a owl:Restriction ; ... ]` axioms) rather than a
        # named class -- those aren't in known_classes, so traversal
        # correctly stops there rather than crashing on a non-class parent.
        if parent is None or parent not in known_classes:
            return 0
        return 1 + depth_of(parent, seen)

    depths = [depth_of(c) for c in classes]

    return {
        'num_classes_declared_local': len(classes),
        'num_subclass_relationships': len(subclass_edges),
        'max_hierarchy_depth': max(depths) if depths else 0,
        'avg_hierarchy_depth': float(np.mean(depths)) if depths else 0.0,
        'num_object_properties': len(object_props),
        'num_datatype_properties': len(datatype_props),
        'num_generic_rdf_properties': len(generic_props),
        'relation_vocabulary_size_declared': len(all_props),
    }


# Vendored copy of the imported upper ontology (see docstring above for why
# this is merged in for SULO's depth traversal only).
sulo_upper_ontology_path = f"{repo_path}/turtles/sulo.ttl"

schema_stats = {
    'Ad-hoc (Base)': schema_metrics(baseline_schema_path),
    'SULO': schema_metrics(sulo_schema_path, upper_ontology_path=sulo_upper_ontology_path),
}

print("NOTE: SULO's hierarchy-depth metrics below are computed with the")
print("vendored upper ontology (turtles/sulo.ttl) merged in for")
print("rdfs:subClassOf traversal, since sulo_kg.ttl's own classes subclass")
print("directly into that imported ontology rather than into each other.")
print("num_classes_declared_local and the property counts still reflect")
print("ONLY this project's own local schema additions, not SULO's full")
print("inherited vocabulary.")
for label, stats in schema_stats.items():
    print(f"\n{label}:")
    for k, v in stats.items():
        print(f"  {k}: {v}")

## Basic Size + Structural Characteristics (Instance-Level)

In [4]:
def instance_structural_metrics(ont, sample_shortest_path_sources=75, high_degree_percentile=99, seed=42):
    """
    Basic size + structural characteristics computed directly from a parsed
    Ontology's edge_index/edge_type (RQ1a). Only the ORIGINAL (forward,
    edge_type % 2 == 0) direction is used -- Ontology doubles every triple
    into a forward+inverse pair for GNN message passing, so halving it back
    out here avoids double-counting nodes/edges/degree.

    Connected components and the sampled average shortest path use scipy's
    sparse graph routines (true all-pairs shortest paths are infeasible at
    multi-million-node scale). Shortest paths are estimated from a random
    sample of source nodes drawn from the largest connected component, one
    source at a time, to keep peak memory to a single [num_nodes] distance
    vector rather than a full sample x num_nodes matrix (which would be tens
    of GB at this graph size).
    """
    t0 = time.time()
    edge_index = ont.edge_index
    edge_type = ont.edge_type
    num_nodes = ont.num_nodes

    forward_mask = (edge_type % 2 == 0)
    fwd_edge_index = edge_index[:, forward_mask]
    num_triples = fwd_edge_index.size(1)

    out_deg = torch.zeros(num_nodes, dtype=torch.long)
    in_deg = torch.zeros(num_nodes, dtype=torch.long)
    out_deg.scatter_add_(0, fwd_edge_index[0], torch.ones(num_triples, dtype=torch.long))
    in_deg.scatter_add_(0, fwd_edge_index[1], torch.ones(num_triples, dtype=torch.long))
    total_deg = out_deg + in_deg
    deg_np = total_deg.numpy()

    density = (2.0 * num_triples) / (num_nodes * (num_nodes - 1)) if num_nodes > 1 else float('nan')
    high_deg_threshold = float(np.percentile(deg_np, high_degree_percentile))
    num_high_degree_nodes = int((deg_np >= high_deg_threshold).sum())

    # "Intermediate" nodes: have both an incoming and an outgoing edge in the
    # original (non-doubled) direction, i.e. they pass information through
    # rather than only originating or only terminating relationships.
    intermediate_mask = (out_deg > 0) & (in_deg > 0)
    num_intermediate_nodes = int(intermediate_mask.sum().item())

    row = fwd_edge_index[0].numpy()
    col = fwd_edge_index[1].numpy()
    data = np.ones(len(row), dtype=np.int8)
    sparse_adj = csr_matrix((data, (row, col)), shape=(num_nodes, num_nodes))

    n_components, component_labels = connected_components(sparse_adj, directed=False, connection='weak')
    component_sizes = np.bincount(component_labels)
    largest_label = int(np.argmax(component_sizes))
    largest_component_frac = float(component_sizes.max() / num_nodes)

    rng = np.random.default_rng(seed)
    in_largest = np.where(component_labels == largest_label)[0]
    sample_size = min(sample_shortest_path_sources, len(in_largest))
    sample_sources = rng.choice(in_largest, size=sample_size, replace=False)

    total_sum, total_count, total_possible = 0.0, 0, 0
    for s in sample_sources:
        dist_row = shortest_path(sparse_adj, method='auto', unweighted=True, directed=False, indices=int(s))
        finite = np.isfinite(dist_row) & (dist_row > 0)  # excludes self (0) and unreachable (inf)
        total_sum += dist_row[finite].sum()
        total_count += int(finite.sum())
        total_possible += (num_nodes - 1)

    avg_shortest_path = total_sum / total_count if total_count else float('nan')
    unreached_frac = 1.0 - (total_count / total_possible) if total_possible else float('nan')

    elapsed = time.time() - t0
    print(f"  instance_structural_metrics done in {elapsed:.1f}s "
          f"({sample_size} shortest-path sources sampled)")

    return {
        'num_nodes': num_nodes,
        'num_edges_triples': num_triples,
        'num_relation_types_observed': len(ont.enum_relations),
        'avg_degree': float(deg_np.mean()),
        'median_degree': float(np.median(deg_np)),
        f'p{high_degree_percentile}_degree_threshold': high_deg_threshold,
        'num_high_degree_nodes': num_high_degree_nodes,
        'graph_density': density,
        'num_connected_components': int(n_components),
        'largest_component_frac_nodes': largest_component_frac,
        'num_intermediate_nodes': num_intermediate_nodes,
        'intermediate_node_frac': num_intermediate_nodes / num_nodes,
        'avg_shortest_path_sampled': avg_shortest_path,
        'shortest_path_sample_size': int(sample_size),
        'shortest_path_unreached_frac': unreached_frac,
    }


instance_stats = {}
for label, entry in KGS.items():
    print(f"Computing structural metrics for {label}...")
    instance_stats[label] = instance_structural_metrics(entry['ont'])
    for k, v in instance_stats[label].items():
        print(f"  {k}: {v}")

Computing structural metrics for Ad-hoc (Base)...
  instance_structural_metrics done in 163.8s (75 shortest-path sources sampled)
  num_nodes: 5125911
  num_edges_triples: 19642035
  num_relation_types_observed: 38
  avg_degree: 7.663822099135158
  median_degree: 6.0
  p99_degree_threshold: 47.0
  num_high_degree_nodes: 51741
  graph_density: 1.49511444780247e-06
  num_connected_components: 2
  largest_component_frac_nodes: 0.9999994147381802
  num_intermediate_nodes: 2794488
  intermediate_node_frac: 0.5451690440977224
  avg_shortest_path_sampled: 3.6020403543541986
  shortest_path_sample_size: 75
  shortest_path_unreached_frac: 5.852619340274146e-07
Computing structural metrics for SULO...
  instance_structural_metrics done in 416.7s (75 shortest-path sources sampled)
  num_nodes: 15009736
  num_edges_triples: 39459059
  num_relation_types_observed: 8
  avg_degree: 5.2577952070576055
  median_degree: 3.0
  p99_degree_threshold: 9.0
  num_high_degree_nodes: 206721
  graph_density: 3.5

## Feature -> Case Proximity & Case Neighbourhood Size

In [5]:
def hop_distance_from_mask(edge_index, num_nodes, source_mask, max_hops=8):
    """
    Exact BFS hop-distance from the NEAREST True node in source_mask to every
    node (unreached nodes within max_hops get -1). Same tensor-frontier
    technique gnn_tools.compute_entity_reuse_stats uses internally, exposed
    here as a per-node distance vector instead of a reached/not-reached mask.
    Uses the full (forward+inverse) edge_index -- doubling doesn't change
    reachability or hop counts, it just traverses each undirected edge from
    both sides, which BFS handles fine.
    """
    src, dst = edge_index
    dist = torch.full((num_nodes,), -1, dtype=torch.long)
    dist[source_mask] = 0
    frontier = source_mask.clone()
    visited = source_mask.clone()
    for hop in range(1, max_hops + 1):
        if not frontier.any():
            break
        active_src = frontier[src]
        active_dst = frontier[dst]
        next_frontier = torch.zeros(num_nodes, dtype=torch.bool)
        next_frontier[dst[active_src]] = True
        next_frontier[src[active_dst]] = True
        next_frontier &= ~visited
        visited |= next_frontier
        dist[next_frontier] = hop
        frontier = next_frontier
    return dist


FEATURE_CONCEPTS = ['CaseDescription', 'TaskDescription', 'CaseSolutionFinalStatement', 'Vehicle', 'Account', 'Task']

proximity_stats = {}
for label, entry in KGS.items():
    ont, data_obj = entry['ont'], entry['data']
    case_mask = get_graph_mask(data_obj, ont, label, 'Case')
    num_cases = int(case_mask.sum().item())

    total_deg = torch.zeros(ont.num_nodes, dtype=torch.long)
    total_deg.scatter_add_(0, data_obj.edge_index[0], torch.ones(data_obj.edge_index.size(1), dtype=torch.long))
    case_neighbourhood_size = float(total_deg[case_mask].float().mean().item()) if num_cases else float('nan')

    case_dist = hop_distance_from_mask(data_obj.edge_index, ont.num_nodes, case_mask, max_hops=8)

    row = {'num_cases': num_cases, 'avg_case_neighbourhood_size': case_neighbourhood_size}
    for concept in FEATURE_CONCEPTS:
        concept_mask = get_graph_mask(data_obj, ont, label, concept)
        n_instances = int(concept_mask.sum().item())
        if n_instances == 0:
            row[f'{concept}_mean_hops_to_case'] = float('nan')
            row[f'{concept}_pct_direct_1hop'] = float('nan')
            continue
        dists = case_dist[concept_mask]
        reached = dists[dists >= 0]
        row[f'{concept}_mean_hops_to_case'] = float(reached.float().mean().item()) if len(reached) else float('nan')
        row[f'{concept}_pct_direct_1hop'] = float((reached == 1).float().mean().item() * 100.0) if len(reached) else float('nan')

    proximity_stats[label] = row
    print(f"\n{label}:")
    for k, v in row.items():
        print(f"  {k}: {v}")

TypeError: Cannot read properties of undefined (reading 'status')

## Assemble Comparison Table

In [6]:
def combine_stats(label):
    combined = {}
    combined.update(schema_stats[label])
    combined.update(instance_stats[label])
    combined.update(proximity_stats[label])
    return combined

comparison_df = pd.DataFrame({label: combine_stats(label) for label in KGS.keys()})
comparison_df.index.name = 'metric'

pd.set_option('display.max_rows', None)
print(comparison_df)

comparison_csv_path = f"{out_dir}rq1a_structural_comparison.csv"
comparison_df.to_csv(comparison_csv_path)
print(f"\nSaved comparison table to: {comparison_csv_path}")

TypeError: Cannot read properties of undefined (reading 'error')

## Visual KG/Subgraph Comparison

In [7]:
def sample_case_subgraph(ont, data_obj, label, num_hops=2, max_nodes=60, seed=42):
    """Small k-hop neighbourhood around a single Case node, for a readable
    side-by-side visual comparison (the full graph has millions of nodes)."""
    case_mask = get_graph_mask(data_obj, ont, label, 'Case')
    case_indices = torch.where(case_mask)[0]
    rng = np.random.default_rng(seed)
    center = int(case_indices[rng.integers(len(case_indices))].item())

    seed_mask = torch.zeros(ont.num_nodes, dtype=torch.bool)
    seed_mask[center] = True
    dist = hop_distance_from_mask(data_obj.edge_index, ont.num_nodes, seed_mask, max_hops=num_hops)

    keep = torch.where(dist >= 0)[0]
    if len(keep) > max_nodes:
        # Always keep the center; sample the rest for readability.
        others = keep[keep != center]
        chosen = rng.choice(others.numpy(), size=max_nodes - 1, replace=False)
        keep = torch.tensor(sorted(set([center]) | set(chosen.tolist())))
    keep_set = set(keep.tolist())

    idx_to_uri = {idx: uri for uri, idx in ont.enum_nodes.items()}

    def short_label(idx):
        uri = idx_to_uri.get(idx, str(idx)).strip('<>').strip('"')
        return uri.split('/')[-1][:20] if '/' in uri else uri[:20]

    G = nx.Graph()
    for idx in keep.tolist():
        G.add_node(idx, label=short_label(idx))

    # Vectorized membership filter first (full edge_index has tens of
    # millions of entries -- a Python-level loop over all of it would be far
    # too slow); only the small filtered result is iterated in Python.
    src, dst = data_obj.edge_index
    keep_mask_full = torch.zeros(ont.num_nodes, dtype=torch.bool)
    keep_mask_full[keep] = True
    edge_keep = keep_mask_full[src] & keep_mask_full[dst]
    sub_src = src[edge_keep].tolist()
    sub_dst = dst[edge_keep].tolist()

    edge_pairs = {tuple(sorted((s, d))) for s, d in zip(sub_src, sub_dst) if s != d}
    G.add_edges_from(edge_pairs)

    return G, center


fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, (label, entry) in zip(axes, KGS.items()):
    G, center = sample_case_subgraph(entry['ont'], entry['data'], label)
    pos = nx.spring_layout(G, seed=42, k=0.6)
    node_colors = ['#d62728' if n == center else '#1f77b4' for n in G.nodes()]
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.4)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=180)
    nx.draw_networkx_labels(G, pos, labels=nx.get_node_attributes(G, 'label'), ax=ax, font_size=6)
    ax.set_title(f"{label}\n({G.number_of_nodes()} nodes, {G.number_of_edges()} edges, "
                 f"2-hop neighbourhood of one Case)")
    ax.axis('off')

plt.tight_layout()
subgraph_fig_path = f"{out_dir}rq1a_subgraph_comparison.png"
plt.savefig(subgraph_fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved subgraph comparison figure to: {subgraph_fig_path}")

TypeError: Cannot read properties of undefined (reading 'error')

## Summary Figure -- Major Structural Differences

In [8]:
summary_metrics = [
    ('num_nodes', 'Nodes'),
    ('num_edges_triples', 'Edges (triples)'),
    ('num_relation_types_observed', 'Relation types (data)'),
    ('relation_vocabulary_size_declared', 'Relation types (schema)'),
    ('num_classes_declared_local', 'Classes (schema, local)'),
    ('max_hierarchy_depth', 'Max hierarchy depth'),
    ('num_subclass_relationships', 'Subclass relationships'),
    ('avg_degree', 'Avg degree'),
    ('graph_density', 'Graph density'),
    ('avg_shortest_path_sampled', 'Avg shortest path (sampled)'),
    ('intermediate_node_frac', 'Intermediate-node fraction'),
    ('avg_case_neighbourhood_size', 'Avg Case neighbourhood size'),
]

labels = list(KGS.keys())
colors = ['#1f77b4', '#d62728']

fig, axes = plt.subplots(3, 4, figsize=(18, 10))
for ax, (metric, title) in zip(axes.flat, summary_metrics):
    values = [combine_stats(l).get(metric, float('nan')) for l in labels]
    ax.bar(labels, values, color=colors)
    ax.set_title(title, fontsize=10)
    ax.tick_params(axis='x', labelsize=8)
    for i, v in enumerate(values):
        if v == v:  # not NaN
            ax.text(i, v, f'{v:,.3g}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
summary_fig_path = f"{out_dir}rq1a_summary_differences.png"
plt.savefig(summary_fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved summary figure to: {summary_fig_path}")

TypeError: Cannot read properties of undefined (reading 'error')